In [ ]:
import json
import pandas as pd

from emu_renewal.constants import ANALYSIS_TYPES, DATA_PATH, OUTPUTS_PATH

In [ ]:
run_id = "58997677"

In [ ]:
def classify_analysis(iso3, analysis, run_path, log_text):
    """Match run.py: complete if store_outputs finished, skipped if MobilityException."""
    if (run_path / iso3 / analysis / "updates.h5").exists():
        return "complete"
    if log_text is None:
        return "no log"
    if f"{analysis} mobility not available" in log_text:
        return "skipped"
    return "not run"


run_path = OUTPUTS_PATH / run_id
countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
status = pd.DataFrame(index=countries, columns=ANALYSIS_TYPES)
for iso3 in countries:
    log_path = run_path / iso3 / "run.log"
    log_text = log_path.read_text() if log_path.exists() else None
    for analysis in ANALYSIS_TYPES:
        status.loc[iso3, analysis] = classify_analysis(iso3, analysis, run_path, log_text)

status

In [ ]:
counts = status.apply(pd.Series.value_counts).fillna(0).astype(int)
counts.loc["total"] = counts.sum()
counts

In [ ]:
not_run = status.isin(["not run"])
rerun_series = not_run.stack()
need_rerun = rerun_series[rerun_series]
rerun_pairs = list(need_rerun.index)
rerun_pairs